In [0]:
%pip install --upgrade "mlflow[databricks]>=3.1"
dbutils.library.restartPython()

%md
# Chapter 6: Evaluating GenAI Applications within MLflow 3+ on Databricks

**Learning Objective:** Understand and apply evaluation techniques for GenAI applications using MLflow.

## Table of Contents
1. [The Modern Evaluation Landscape in MLflow](#modern-evaluation)
2. [Creating & Managing Evaluation Datasets](#evaluation-datasets)
3. [Scores](#scores)
4. [Human Feedback](#human-feedback)
5. [Evaluation Runs](#evaluation-runs)
6. [Best Practices](#best-practices)

---

## 1. The Modern Evaluation Landscape in MLflow {#modern-evaluation}

GenAI applications represent a paradigm shift from traditional ML models. Unlike classic models that produce structured, deterministic outputs, GenAI systems generate natural language responses that must be evaluated for qualities like relevance, helpfulness, factual accuracy, and user satisfaction.

### Why Traditional Metrics Miss GenAI Application Behaviour

Traditional ML evaluation metrics (accuracy, precision, recall, F1-score) were designed for classification and regression tasks with clear ground truth labels. These metrics fall short for GenAI applications because:

1. **Deterministic vs. Generative Outputs**: Traditional models produce fixed outputs for given inputs, while GenAI models generate variable, creative responses
2. **Structured vs. Unstructured Data**: Traditional metrics work with numerical or categorical outputs, not natural language text
3. **Single Correct Answer vs. Multiple Valid Responses**: GenAI tasks often have many acceptable answers, making binary accuracy insufficient
4. **Context and Nuance**: Traditional metrics don't capture semantic meaning, tone, helpfulness, or user experience
5. **Safety and Ethics**: GenAI outputs must be evaluated for harmful content, bias, and policy compliance

### What are the Components to Evaluate

GenAI applications typically consist of multiple components that require different evaluation approaches:

1. **Retrieval Components**: 
   - Retrieval accuracy and relevance
   - Document ranking quality
   - Coverage of relevant information

2. **Generation Components**:
   - Factual correctness and groundedness
   - Relevance to user query
   - Coherence and fluency
   - Tone and style appropriateness

3. **End-to-End System**:
   - User experience and satisfaction
   - Task completion effectiveness
   - Safety and policy compliance
   - Latency and performance

4. **Business Logic**:
   - Adherence to guidelines and policies
   - Consistency across similar queries
   - Integration with downstream systems

### Evaluation Modes: Direct Evaluation vs Answer Sheet Evaluation

**Direct Evaluation:**
- Assesses model outputs directly against criteria or guidelines
- Uses LLM judges to evaluate qualities like helpfulness, relevance, safety
- Suitable for open-ended tasks without single correct answers
- Examples: Chatbot responses, creative writing, summarization

**Answer Sheet Evaluation:**
- Compares model outputs to curated reference answers
- Uses exact match, semantic similarity, or custom comparison functions
- Suitable for tasks with clear correct answers
- Examples: Question answering, fact extraction, classification

## Use Case: Unity Airways Customer Service Agent

This notebook demonstrates best practices for evaluating a GenAI-powered customer service agent for Unity Airways (a fictional airline). The agent helps customers with:
- Flight bookings and modifications
- Policy questions (baggage, refunds, cancellations)
- General customer support inquiries

We'll evaluate using both structured booking data and unstructured FAQ/QA datasets, demonstrating the full spectrum of GenAI evaluation techniques in MLflow 3+.

%md


In [0]:
# Define catalog and schema for Unity Airways datasets
CATALOG = "workspace"
SCHEMA = "unity_airways"

# Example usage for table reference:
# spark.read.table(f"{CATALOG}.{SCHEMA}.booking_records_dataset")
# This will be used throughout the notebook for consistency.

%md
## 2. Creating & Managing Evaluation Datasets {#evaluation-datasets}

Evaluation datasets are the foundation of GenAI application testing. MLflow 3+ provides powerful tools for creating, managing, and versioning evaluation datasets that enable systematic testing and continuous improvement.

### Dataset Types and Sources

**1. Production Trace Datasets**
- Built from real user interactions captured by MLflow Tracing
- Provides authentic user scenarios and edge cases
- Enables testing against actual production patterns

**2. Curated Test Datasets**
- Manually created examples targeting specific features or edge cases
- Ground truth answers for answer sheet evaluation
- Domain expert validated responses

**3. Synthetic Datasets**
- Generated using LLMs to expand test coverage
- Useful for testing rare scenarios or adversarial cases
- Can simulate different user personas and interaction styles

### Building Evaluation Datasets in MLflow 3+

MLflow 3+ provides the `mlflow.genai.datasets` API for creating managed evaluation datasets with versioning and lineage tracking.

```python
import mlflow.genai.datasets

# Create a new evaluation dataset
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name="catalog.schema.dataset_name",
    name="Unity Airways Customer Service Evaluation",
    description="Evaluation dataset for Unity Airways customer service agent"
)
```

### Our Unity Airways Datasets

We'll work with three complementary datasets:
- **Booking Records**: Structured data for testing booking-related queries
- **FAQ Dataset**: Common customer questions with approved answers
- **QA Dataset**: Curated question-answer pairs for direct evaluation

Let's explore these datasets to understand their structure and content.

In [0]:
booking_records_df = spark.read.table(f"{CATALOG}.{SCHEMA}.booking_records_dataset")
display(booking_records_df.limit(10))

In [0]:
faq_df = spark.read.table(f"{CATALOG}.{SCHEMA}.faq_dataset")
display(faq_df.limit(10))

In [0]:
qa_df = spark.read.table(f"{CATALOG}.{SCHEMA}.qa_dataset")
display(qa_df.limit(10))

%md
### Approaches to Building Evaluation Datasets

MLflow 3+ offers several flexible approaches to construct evaluation datasets:

#### Approach 1: Build from Existing Traces

One of the most effective ways to build relevant evaluation datasets is by curating examples from your application's historical interactions captured by MLflow Tracing.

```python
import mlflow
import time

# Search for traces from the last hour
one_hour_ago = int((time.time() - 60 * 60) * 1000)

traces = mlflow.search_traces(
    filter_string=f"attributes.timestamp_ms > {one_hour_ago} AND "
                 f"attributes.status = 'OK'",
    order_by=["attributes.timestamp_ms DESC"],
    max_results=100
)

# Add traces to evaluation dataset
eval_dataset.merge_records(traces)
```

#### Approach 2: Build from Scratch or Import Existing

You can import existing datasets or create examples from scratch. Data must match the evaluation dataset schema:

```python
evaluation_examples = [
    {
        "inputs": {"question": "What is the baggage allowance for international flights?"},
        "expected": {
            "expected_response": "For international flights, you can bring one carry-on bag (22x14x9 inches) and one personal item. Checked baggage allowance varies by fare type.",
            "expected_categories": ["baggage", "international", "policy"]
        }
    },
    {
        "inputs": {"question": "How do I cancel my flight?"},
        "expected": {
            "expected_response": "You can cancel your flight online through Manage My Booking, by calling customer service, or at the airport. Cancellation fees may apply.",
            "expected_categories": ["cancellation", "booking", "policy"]
        }
    }
]

eval_dataset.merge_records(evaluation_examples)
```

#### Approach 3: Synthesize Evaluation Sets

Generate synthetic data to expand testing coverage and create diverse scenarios:

```python
from mlflow.genai.datasets import synthesize_dataset

# Generate synthetic customer service scenarios
synthetic_dataset = synthesize_dataset(
    base_examples=evaluation_examples,
    num_examples=50,
    persona_variations=["frustrated customer", "first-time flyer", "business traveler"],
    scenario_types=["booking", "cancellation", "policy inquiry", "complaint"]
)

eval_dataset.merge_records(synthetic_dataset)
```

#### Approach 4: Domain Expert Labels

Leverage feedback from domain experts captured in MLflow Labeling Sessions:

```python
import mlflow.genai.labeling as labeling

# Get labeling sessions
labeling_sessions = labeling.get_labeling_sessions()

# Sync labeled data to evaluation dataset
for session in labeling_sessions:
    if session.name == "Unity Airways Customer Service Review":
        session.sync(dataset_name="catalog.schema.unity_airways_eval")
```

### Creating MLflow Evaluation Datasets

Let's demonstrate creating a managed evaluation dataset from our Unity Airways data:

```python
import mlflow.genai.datasets

# Create evaluation dataset
eval_dataset = mlflow.genai.datasets.create_dataset(
    uc_table_name=f"{CATALOG}.{SCHEMA}.unity_airways_evaluation_dataset",
    name="Unity Airways Customer Service Evaluation",
    description="Comprehensive evaluation dataset for Unity Airways customer service agent"
)
```

For more details, see: [Build Evaluation Dataset Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset)

In [0]:
# Install textblob for sentiment analysis
import pandas as pd

qa_sample = qa_df.limit(100).toPandas()
qa_sample['prediction'] = qa_sample['answer']  # Simulate chatbot output

# Accuracy: 1 if prediction matches reference, else 0
qa_sample['accuracy'] = (qa_sample['prediction'] == qa_sample['answer']).astype(int)

# Relevance: simple keyword overlap
qa_sample['relevance'] = qa_sample.apply(
    lambda row: len(set(row['prediction'].split()) & set(row['question'].split())) / max(len(set(row['question'].split())), 1),
    axis=1
)

display(qa_sample[['question', 'answer', 'prediction', 'accuracy', 'relevance']].head(10))

In [0]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Create interactive histograms for score distributions using Plotly
metrics_to_plot = ['accuracy', 'relevance', 'length_score']
existing_metrics = [metric for metric in metrics_to_plot if metric in qa_sample.columns]

if existing_metrics:
    # Create subplots for multiple metrics
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f"Distribution of {metric.capitalize()} Scores" for metric in existing_metrics[:4]]
    )
    
    colors = ['blue', 'green', 'red', 'orange']
    
    for i, metric in enumerate(existing_metrics[:4]):
        row = (i // 2) + 1
        col = (i % 2) + 1
        
        fig.add_trace(
            go.Histogram(
                x=qa_sample[metric].dropna(),
                nbinsx=10,
                name=metric.capitalize(),
                marker_color=colors[i],
                opacity=0.7,
                showlegend=False
            ),
            row=row, col=col
        )
        
        fig.update_xaxes(title_text=f"{metric.capitalize()} Score", row=row, col=col)
        fig.update_yaxes(title_text="Count", row=row, col=col)
    
    fig.update_layout(
        height=600,
        title_text="Score Distributions - Interactive Visualization",
        showlegend=False
    )
    
    fig.show()
else:
    print("No score metrics found in the data to visualize.")

%md
## 6.4 Human Feedback: Collecting and Analyzing User Responses

Human feedback is essential for evaluating GenAI applications, as it provides insights into user satisfaction, perceived helpfulness, and areas for improvement. In MLflow, feedback can be collected and analyzed to refine chatbot performance and guide future development.

In this section, we will:
- Simulate collecting end-user feedback for Unity Airways chatbot responses
- Analyze feedback data
- Visualize feedback distribution
- Discuss best practices for feedback collection and analysis

%md
### Best Practices for Human Feedback in GenAI Evaluation

- **Integrate feedback collection into chatbot UI** to capture real user sentiment and suggestions.
- **Use structured feedback forms** (e.g., thumbs up/down, star ratings, free text) for actionable insights.
- **Analyze feedback regularly** to identify trends, common issues, and opportunities for improvement.
- **Correlate feedback with model outputs** to understand which responses drive positive or negative user experiences.
- **Leverage MLflow tracking** to log feedback data alongside evaluation runs for comprehensive analysis.

Human feedback is a critical component in building trustworthy and effective GenAI applications.

%md
### Best Practices for Evaluation Runs in GenAI Projects

- **Log evaluation results in MLflow** for traceability and reproducibility
- **Track key metrics** (accuracy, relevance, sentiment, custom scores, feedback) for each run
- **Compare runs over time** to monitor improvements and regressions
- **Use MLflow artifacts** to store sample outputs, charts, and feedback data
- **Automate evaluation runs** as part of your model development and deployment workflow

For full MLflow GenAI evaluation support, refer to the [MLflow GenAI evaluation documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/).

%md
## 6.6 Best Practices for GenAI Evaluation in MLflow and Databricks

- **Use both structured and unstructured datasets** to comprehensively evaluate GenAI applications.
- **Leverage MLflow 3+ GenAI evaluation features** for advanced scoring, custom judges, and feedback integration.
- **Apply a mix of LLM-based and code-based metrics** to capture accuracy, relevance, sentiment, and business logic.
- **Integrate human feedback** to understand real user experience and drive improvements.
- **Track and compare evaluation runs in MLflow** for reproducibility and continuous improvement.
- **Visualize results and feedback** to identify trends and actionable insights.
- **Automate evaluation as part of your development workflow** to ensure consistent quality.

By following these best practices, you can build robust, user-centric GenAI solutions that deliver real value in production environments.

%md
### Building MLflow Evaluation Datasets from Unity Airways Tables

We will use MLflow 3's `mlflow.genai.monitor.EvaluationDataset` API to build evaluation datasets from the Unity Airways tables. This enables standardized evaluation workflows for GenAI applications.

Reference: [Build Evaluation Dataset (Databricks Documentation)](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset)

We'll demonstrate this for all three tables: booking records, FAQ, and QA. For efficiency, we use small samples (100 rows each).

%md
## 3. Scores {#scores}

Scoring is the heart of GenAI evaluation. MLflow 3+ provides a comprehensive framework for assessing GenAI applications using both automated and human-informed evaluation methods.

### How Scores Work

Scores in MLflow GenAI evaluation are functions that take model inputs and outputs and return a quantitative or qualitative assessment. They can be:

1. **Numerical scores** (0-1, 1-5, etc.) for quantitative metrics
2. **Boolean scores** (pass/fail) for binary criteria
3. **Categorical scores** (good/fair/poor) for qualitative assessments
4. **Structured feedback** with scores and rationales

### Types of Scorers

#### 1. LLM-Based Scorers

Use large language models as judges to evaluate outputs for complex qualities that require understanding of context, semantics, and nuance.

**Advantages:**
- Can evaluate subjective qualities (helpfulness, tone, coherence)
- Understand context and nuance
- Provide detailed rationales
- Scale to evaluate large datasets

**Disadvantages:**
- Can be inconsistent
- Require careful prompt engineering
- May have biases
- Higher latency and cost

#### 2. Code-Based Scorers

Use deterministic Python functions to evaluate outputs based on specific rules, patterns, or calculations.

**Advantages:**
- Deterministic and consistent
- Fast execution
- No additional LLM costs
- Easy to debug and modify

**Disadvantages:**
- Limited to rule-based evaluation
- Cannot assess subjective qualities
- May miss nuanced cases
- Require manual rule definition

### Using LLM-Based Scorers

MLflow 3+ provides both predefined and custom LLM-based scorers:

#### Predefined LLM Scorers

```python
from mlflow.genai.scorers import (
    RetrievalGroundedness,    # Checks if response is grounded in retrieved context
    RelevanceToQuery,         # Evaluates relevance to user query  
    Safety,                   # Detects harmful or inappropriate content
    AnswerCorrectness,        # Compares against ground truth answers
    Faithfulness,             # Checks factual consistency
    AnswerRelevance           # Evaluates answer relevance
)

# Use predefined scorers
scorers = [
    RetrievalGroundedness(),
    RelevanceToQuery(), 
    Safety(),
    AnswerCorrectness()
]
```

For complete list and details, see: [Predefined Judge Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/predefined-judge-scorers)

#### Custom LLM Scorers

**Guidelines-Based Judges:**
Evaluate outputs against specific guidelines or policies.

```python
from mlflow.genai.scorers import Guidelines

# Custom guidelines-based scorer
professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="The response must use professional, courteous language appropriate for customer service. Avoid casual expressions, slang, or overly informal tone."
)

brand_compliance_scorer = Guidelines(
    name="brand_compliance", 
    guidelines="The response must follow Unity Airways brand guidelines: mention the airline name when relevant, use positive language, and maintain helpful tone."
)
```

**Prompt-Based Judges:**
Create custom evaluation logic using prompt engineering.

```python
from mlflow.genai.scorers import PromptBasedJudge

# Custom prompt-based scorer
completeness_scorer = PromptBasedJudge(
    name="response_completeness",
    prompt="""
    Evaluate if the customer service response completely addresses the customer's question.
    
    Customer Question: {question}
    Agent Response: {response}
    
    Consider:
    1. Does the response directly answer the question?
    2. Are all parts of multi-part questions addressed?
    3. Is sufficient detail provided?
    4. Are next steps clearly explained?
    
    Rate on a scale of 1-5 where:
    1 = Completely inadequate, major parts of question ignored
    2 = Partially addresses question but missing important elements  
    3 = Addresses main question but lacks some detail
    4 = Addresses question well with good detail
    5 = Completely and thoroughly addresses all aspects
    
    Provide your rating and brief explanation.
    """
)
```

For more details, see: 
- [Custom Judge Guidelines](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/meets-guidelines)
- [Create Prompt Judge](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-judge/create-prompt-judge)

### Using Code-Based Scorers

Code-based scorers provide deterministic evaluation using Python functions:

```python
from mlflow.genai.scorers import CodeBasedScorer

def response_length_scorer(prediction, **kwargs):
    """Score based on response length appropriateness."""
    length = len(prediction.split())
    if length < 10:
        return {"score": 0, "justification": "Response too short"}
    elif length > 200:
        return {"score": 0, "justification": "Response too long"}
    else:
        return {"score": 1, "justification": "Appropriate length"}

def contact_info_scorer(prediction, **kwargs):
    """Check if response includes contact information when appropriate."""
    contact_keywords = ["call", "phone", "email", "contact"]
    if any(keyword in prediction.lower() for keyword in contact_keywords):
        if "1-800" in prediction or "@unityairways.com" in prediction:
            return {"score": 1, "justification": "Includes proper contact info"}
        else:
            return {"score": 0, "justification": "Mentions contact but no specific info"}
    return {"score": 0.5, "justification": "No contact info mentioned"}

# Create code-based scorers
length_scorer = CodeBasedScorer(name="response_length", func=response_length_scorer)
contact_scorer = CodeBasedScorer(name="includes_contact_info", func=contact_info_scorer)
```

For more details, see: [Custom Scorers](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/custom-scorers)

### How to Evaluate Different Components

Different parts of your GenAI application require different evaluation approaches:

#### Retrieval Components
```python
retrieval_scorers = [
    RetrievalGroundedness(),  # Are responses grounded in retrieved docs?
    RelevanceToQuery(),       # Do retrieved docs match the query?
]
```

#### Generation Components  
```python
generation_scorers = [
    AnswerCorrectness(),      # Factual accuracy
    Faithfulness(),           # Consistency with source material
    Guidelines(name="tone", guidelines="Professional and helpful tone"),
]
```

#### End-to-End System
```python
system_scorers = [
    Safety(),                 # Content safety
    Guidelines(name="completeness", guidelines="Complete and actionable response"),
    CodeBasedScorer(name="latency", func=latency_checker),
]
```

Let's now demonstrate these concepts with our Unity Airways customer service agent.

%md
### How to Use MLflow 3+ LLM-Based Scorers and Custom Judges

If your workspace supports MLflow 3+ GenAI evaluation APIs, you can use built-in and custom LLM scorers for advanced evaluation:

**Predefined LLM Scorers (Python):**
```python
from mlflow.genai.monitor import PredefinedJudges

# Example: Use accuracy and relevance judges
judges = [PredefinedJudges.ACCURACY, PredefinedJudges.RELEVANCE]
results = qa_eval_dataset.score(judges=judges)
```

**Custom LLM Judges (Guidelines/Prompt-Based):**
```python
from mlflow.genai.monitor import GuidelinesBasedJudge, PromptBasedJudge

# Example: Guidelines-based judge
guidelines = "Answers must be polite, factual, and concise."
custom_judge = GuidelinesBasedJudge(guidelines=guidelines)
results = qa_eval_dataset.score(judges=[custom_judge])
```

For more details, see: [MLflow GenAI Evaluation Scoring](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/scoring.html)

If MLflow 3+ APIs are not available, use Python-based scoring as shown below.

%md
## Convert Human Feedback Visualizations to Plotly
There are likely additional matplotlib-based visualizations for human feedback distribution in later cells. I will add a Plotly-based version for user feedback and feedback score distributions, similar to the original matplotlib code.

## Practical Implementation: Unity Airways Customer Service Agent

Let's now implement a complete GenAI evaluation workflow using our Unity Airways customer service agent. This section demonstrates all the concepts covered in this chapter with working code.

### Step 1: Create a Customer Service Agent

First, let's create a simple customer service agent that can answer common questions about Unity Airways policies and services.


In [0]:
# Unity Airways Customer Service Agent Implementation
import mlflow
import mlflow.genai.datasets
from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines, CodeBasedScorer
import pandas as pd
import time
from typing import Dict, List

# Enable MLflow tracing
mlflow.openai.autolog()

# Unity Airways Knowledge Base (simplified for demo)
UNITY_AIRWAYS_KB = {
    "baggage": {
        "carry_on": "One carry-on bag (22x14x9 inches) and one personal item allowed",
        "checked": "First checked bag free for premium customers, $30 for economy",
        "weight_limit": "50 lbs for checked bags, no weight limit for carry-on"
    },
    "cancellation": {
        "policy": "Cancel within 24 hours for full refund. After 24h, fees apply based on fare type",
        "fees": "Economy: $200, Premium: $100, First Class: No fee",
        "process": "Cancel online, by phone 1-800-UNITY-AIR, or at airport"
    },
    "booking": {
        "online": "Book at unityairways.com or mobile app",
        "phone": "Call 1-800-UNITY-AIR for assistance",
        "changes": "Changes allowed up to 2 hours before departure"
    }
}

@mlflow.trace
def unity_airways_customer_service_agent(customer_question: str) -> Dict[str, str]:
    """
    Unity Airways customer service agent that answers common questions.
    This is a simplified rule-based agent for demonstration purposes.
    """
    question_lower = customer_question.lower()
    
    # Simple keyword-based routing
    if any(word in question_lower for word in ["baggage", "bag", "luggage"]):
        if "carry" in question_lower or "cabin" in question_lower:
            response = UNITY_AIRWAYS_KB["baggage"]["carry_on"]
        elif "checked" in question_lower:
            response = UNITY_AIRWAYS_KB["baggage"]["checked"]
        elif "weight" in question_lower:
            response = UNITY_AIRWAYS_KB["baggage"]["weight_limit"]
        else:
            response = "For baggage information: " + UNITY_AIRWAYS_KB["baggage"]["carry_on"]
    
    elif any(word in question_lower for word in ["cancel", "cancellation", "refund"]):
        if "fee" in question_lower or "cost" in question_lower:
            response = UNITY_AIRWAYS_KB["cancellation"]["fees"]
        elif "how" in question_lower or "process" in question_lower:
            response = UNITY_AIRWAYS_KB["cancellation"]["process"]
        else:
            response = UNITY_AIRWAYS_KB["cancellation"]["policy"]
    
    elif any(word in question_lower for word in ["book", "booking", "reservation"]):
        if "change" in question_lower or "modify" in question_lower:
            response = UNITY_AIRWAYS_KB["booking"]["changes"]
        elif "phone" in question_lower:
            response = UNITY_AIRWAYS_KB["booking"]["phone"]
        else:
            response = UNITY_AIRWAYS_KB["booking"]["online"]
    
    else:
        response = "Thank you for contacting Unity Airways. For immediate assistance, please call 1-800-UNITY-AIR or visit our website at unityairways.com"
    
    # Add helpful closing
    response += " Is there anything else I can help you with today?"
    
    return {"response": response}

# Test the agent
test_question = "What's the baggage policy for carry-on items?"
result = unity_airways_customer_service_agent(test_question)
print(f"Question: {test_question}")
print(f"Response: {result['response']}")


### Step 2: Create Comprehensive Evaluation Dataset

Now let's create a comprehensive evaluation dataset that covers different types of customer service scenarios.


In [0]:
# Create comprehensive evaluation dataset
evaluation_examples = [
    # Baggage policy questions
    {
        "inputs": {"customer_question": "What can I bring in my carry-on bag?"},
        "expected": {
            "expected_response": "One carry-on bag (22x14x9 inches) and one personal item allowed",
            "category": "baggage",
            "complexity": "simple"
        }
    },
    {
        "inputs": {"customer_question": "How much does it cost to check a bag?"},
        "expected": {
            "expected_response": "First checked bag free for premium customers, $30 for economy",
            "category": "baggage", 
            "complexity": "simple"
        }
    },
    {
        "inputs": {"customer_question": "What's the weight limit for my luggage?"},
        "expected": {
            "expected_response": "50 lbs for checked bags, no weight limit for carry-on",
            "category": "baggage",
            "complexity": "simple"
        }
    },
    
    # Cancellation policy questions
    {
        "inputs": {"customer_question": "Can I cancel my flight and get a refund?"},
        "expected": {
            "expected_response": "Cancel within 24 hours for full refund. After 24h, fees apply based on fare type",
            "category": "cancellation",
            "complexity": "medium"
        }
    },
    {
        "inputs": {"customer_question": "How much will it cost to cancel my economy ticket?"},
        "expected": {
            "expected_response": "Economy: $200, Premium: $100, First Class: No fee",
            "category": "cancellation",
            "complexity": "medium"
        }
    },
    {
        "inputs": {"customer_question": "How do I cancel my reservation?"},
        "expected": {
            "expected_response": "Cancel online, by phone 1-800-UNITY-AIR, or at airport",
            "category": "cancellation",
            "complexity": "simple"
        }
    },
    
    # Booking questions
    {
        "inputs": {"customer_question": "Where can I book a flight?"},
        "expected": {
            "expected_response": "Book at unityairways.com or mobile app",
            "category": "booking",
            "complexity": "simple"
        }
    },
    {
        "inputs": {"customer_question": "Can I change my flight after booking?"},
        "expected": {
            "expected_response": "Changes allowed up to 2 hours before departure",
            "category": "booking",
            "complexity": "medium"
        }
    },
    
    # Edge cases and complex questions
    {
        "inputs": {"customer_question": "I need help with my flight but I don't know what specifically"},
        "expected": {
            "expected_response": "Thank you for contacting Unity Airways. For immediate assistance, please call 1-800-UNITY-AIR or visit our website at unityairways.com",
            "category": "general",
            "complexity": "complex"
        }
    },
    {
        "inputs": {"customer_question": "What's your policy on emotional support animals?"},
        "expected": {
            "expected_response": "Thank you for contacting Unity Airways. For immediate assistance, please call 1-800-UNITY-AIR or visit our website at unityairways.com",
            "category": "general", 
            "complexity": "complex"
        }
    }
]

# Convert to DataFrame for easier manipulation
eval_df = pd.DataFrame(evaluation_examples)
print(f"Created evaluation dataset with {len(eval_df)} examples")
print("\nDataset distribution by category:")
print(eval_df['expected'].apply(lambda x: x['category']).value_counts())
print("\nDataset distribution by complexity:")
print(eval_df['expected'].apply(lambda x: x['complexity']).value_counts())

# Display sample
print("\nSample evaluation example:")
sample = eval_df.iloc[0]
print(f"Question: {sample['inputs']['customer_question']}")
print(f"Expected: {sample['expected']['expected_response']}")
print(f"Category: {sample['expected']['category']}")
print(f"Complexity: {sample['expected']['complexity']}")


### Step 3: Define Comprehensive Scorers

Let's create a mix of LLM-based and code-based scorers to evaluate our customer service agent comprehensively.


In [0]:
# Define comprehensive scorers for Unity Airways customer service evaluation

# 1. LLM-Based Scorers
from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines

# Custom Guidelines-Based Scorers
professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="""
    The customer service response must use professional, courteous language appropriate for airline customer service.
    Requirements:
    - Use polite and respectful language
    - Avoid casual expressions or slang
    - Maintain helpful and solution-oriented tone
    - Include appropriate greetings/closings when relevant
    """
)

brand_compliance_scorer = Guidelines(
    name="brand_compliance",
    guidelines="""
    The response must follow Unity Airways brand guidelines:
    - Mention Unity Airways when appropriate
    - Use consistent contact information (1-800-UNITY-AIR, unityairways.com)
    - Maintain positive, customer-focused messaging
    - Provide specific, actionable information when possible
    """
)

completeness_scorer = Guidelines(
    name="response_completeness", 
    guidelines="""
    The customer service response must completely address the customer's question:
    - Directly answer the specific question asked
    - Provide all relevant details mentioned in the expected response
    - Include next steps or additional resources when appropriate
    - Avoid generic responses when specific information is requested
    """
)

# 2. Code-Based Scorers

def response_length_checker(prediction, **kwargs):
    """Check if response length is appropriate (not too short or too long)."""
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    word_count = len(response.split())
    
    if word_count < 5:
        return {"score": 0, "justification": f"Response too short ({word_count} words)"}
    elif word_count > 100:
        return {"score": 0.5, "justification": f"Response quite long ({word_count} words)"}
    else:
        return {"score": 1, "justification": f"Appropriate length ({word_count} words)"}

def contact_info_checker(prediction, **kwargs):
    """Check if response includes appropriate Unity Airways contact information."""
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    response_lower = response.lower()
    
    has_phone = "1-800-unity-air" in response_lower
    has_website = "unityairways.com" in response_lower
    mentions_contact = any(word in response_lower for word in ["call", "phone", "website", "visit"])
    
    if has_phone and has_website:
        return {"score": 1, "justification": "Includes both phone and website"}
    elif has_phone or has_website:
        return {"score": 0.8, "justification": "Includes one form of contact info"}
    elif mentions_contact:
        return {"score": 0.3, "justification": "Mentions contacting but no specific info"}
    else:
        return {"score": 0, "justification": "No contact information provided"}

def policy_accuracy_checker(prediction, expected=None, **kwargs):
    """Check if the response contains accurate policy information."""
    if not expected:
        return {"score": 0.5, "justification": "No expected response to compare against"}
    
    response = prediction if isinstance(prediction, str) else prediction.get('response', '')
    expected_response = expected.get('expected_response', '') if isinstance(expected, dict) else str(expected)
    
    # Simple keyword overlap check
    response_words = set(response.lower().split())
    expected_words = set(expected_response.lower().split())
    
    # Remove common words
    common_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are'}
    response_words -= common_words
    expected_words -= common_words
    
    if not expected_words:
        return {"score": 0.5, "justification": "No meaningful expected words to compare"}
    
    overlap = len(response_words & expected_words) / len(expected_words)
    
    if overlap >= 0.7:
        return {"score": 1, "justification": f"High accuracy ({overlap:.2f} keyword overlap)"}
    elif overlap >= 0.4:
        return {"score": 0.7, "justification": f"Good accuracy ({overlap:.2f} keyword overlap)"}
    elif overlap >= 0.2:
        return {"score": 0.4, "justification": f"Partial accuracy ({overlap:.2f} keyword overlap)"}
    else:
        return {"score": 0, "justification": f"Low accuracy ({overlap:.2f} keyword overlap)"}

# Create code-based scorers
length_scorer = CodeBasedScorer(name="response_length", func=response_length_checker)
contact_scorer = CodeBasedScorer(name="includes_contact_info", func=contact_info_checker)
accuracy_scorer = CodeBasedScorer(name="policy_accuracy", func=policy_accuracy_checker)

# Combine all scorers
unity_airways_scorers = [
    # LLM-based scorers
    RelevanceToQuery(),
    Safety(),
    professional_tone_scorer,
    brand_compliance_scorer,
    completeness_scorer,
    
    # Code-based scorers  
    length_scorer,
    contact_scorer,
    accuracy_scorer
]

print(f"Created {len(unity_airways_scorers)} scorers for Unity Airways evaluation:")
for scorer in unity_airways_scorers:
    print(f"- {scorer.name}: {'LLM-based' if hasattr(scorer, 'guidelines') or scorer.name in ['relevance_to_query', 'safety'] else 'Code-based'}")

# Test a scorer
test_response = "One carry-on bag (22x14x9 inches) and one personal item allowed. Is there anything else I can help you with today?"
test_expected = {"expected_response": "One carry-on bag (22x14x9 inches) and one personal item allowed"}

accuracy_result = policy_accuracy_checker(test_response, test_expected)
print(f"\nTest scorer result: {accuracy_result}")


### Step 4: Run Complete Evaluation and Analysis

Now let's run a complete evaluation of our Unity Airways customer service agent and analyze the results comprehensively.


In [0]:
# Step 4: Run Complete Evaluation

# Note: This demonstrates the evaluation workflow. In practice, you would use:
# eval_results = mlflow.genai.evaluate(
#     data=eval_df,
#     predict_fn=unity_airways_customer_service_agent,
#     scorers=unity_airways_scorers,
#     run_name="Unity Airways Customer Service Evaluation v1.0"
# )

# For this demonstration, we'll simulate the evaluation process
print("Running Unity Airways Customer Service Evaluation...")
print("=" * 60)

# Simulate running evaluation on our dataset
evaluation_results = []

for idx, row in eval_df.iterrows():
    question = row['inputs']['customer_question']
    expected = row['expected']
    
    print(f"\nEvaluating example {idx + 1}/{len(eval_df)}")
    print(f"Question: {question}")
    
    # Get agent response
    agent_response = unity_airways_customer_service_agent(question)
    response_text = agent_response['response']
    
    print(f"Response: {response_text[:100]}...")
    
    # Apply code-based scorers (simulating what MLflow would do)
    scores = {}
    
    # Length scorer
    length_result = response_length_checker(response_text)
    scores['response_length'] = length_result['score']
    
    # Contact info scorer  
    contact_result = contact_info_checker(response_text)
    scores['includes_contact_info'] = contact_result['score']
    
    # Policy accuracy scorer
    accuracy_result = policy_accuracy_checker(response_text, expected)
    scores['policy_accuracy'] = accuracy_result['score']
    
    # Store results
    evaluation_results.append({
        'question': question,
        'response': response_text,
        'expected_response': expected['expected_response'],
        'category': expected['category'],
        'complexity': expected['complexity'],
        **scores,
        'length_justification': length_result['justification'],
        'contact_justification': contact_result['justification'],
        'accuracy_justification': accuracy_result['justification']
    })
    
    print(f"Scores: Length={scores['response_length']:.2f}, Contact={scores['includes_contact_info']:.2f}, Accuracy={scores['policy_accuracy']:.2f}")

# Convert results to DataFrame for analysis
results_df = pd.DataFrame(evaluation_results)

print(f"\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)
print(f"Evaluated {len(results_df)} examples")

# Calculate overall metrics
overall_metrics = {
    'avg_response_length': results_df['response_length'].mean(),
    'avg_contact_info': results_df['includes_contact_info'].mean(), 
    'avg_policy_accuracy': results_df['policy_accuracy'].mean(),
    'overall_score': results_df[['response_length', 'includes_contact_info', 'policy_accuracy']].mean().mean()
}

print(f"\nOVERALL METRICS:")
for metric, value in overall_metrics.items():
    print(f"- {metric}: {value:.3f}")

# Performance by category
print(f"\nPERFORMANCE BY CATEGORY:")
category_performance = results_df.groupby('category')[['response_length', 'includes_contact_info', 'policy_accuracy']].mean()
display(category_performance)

# Performance by complexity
print(f"\nPERFORMANCE BY COMPLEXITY:")
complexity_performance = results_df.groupby('complexity')[['response_length', 'includes_contact_info', 'policy_accuracy']].mean()
display(complexity_performance)

# Identify failures and improvement opportunities
print(f"\nFAILURE ANALYSIS:")
failures = results_df[results_df['policy_accuracy'] < 0.5]
if len(failures) > 0:
    print(f"Found {len(failures)} examples with low policy accuracy:")
    for idx, failure in failures.iterrows():
        print(f"- {failure['category']}: {failure['question'][:50]}... (Score: {failure['policy_accuracy']:.2f})")
        print(f"  Reason: {failure['accuracy_justification']}")
else:
    print("No significant policy accuracy failures found!")

# Success analysis
print(f"\nSUCCESS ANALYSIS:")
successes = results_df[results_df['policy_accuracy'] >= 0.8]
print(f"Found {len(successes)} examples with high policy accuracy:")
success_categories = successes['category'].value_counts()
print("Success distribution by category:")
for category, count in success_categories.items():
    print(f"- {category}: {count} examples")

print(f"\nDETAILED RESULTS SAMPLE:")
display(results_df[['question', 'category', 'response_length', 'includes_contact_info', 'policy_accuracy']].head())


### Step 5: Simulate Human Feedback Collection

In a real implementation, you would collect human feedback through your application UI or review processes. Here's how you might simulate and analyze feedback data:


In [0]:
# Simulate human feedback collection
import numpy as np
import matplotlib.pyplot as plt

# Simulate feedback based on policy accuracy (higher accuracy -> better feedback)
np.random.seed(42)

feedback_data = []
for idx, row in results_df.iterrows():
    # Simulate feedback probability based on policy accuracy
    accuracy = row['policy_accuracy']
    
    # Higher accuracy leads to better feedback probability
    positive_prob = 0.3 + 0.6 * accuracy  # 30% base + up to 60% based on accuracy
    negative_prob = 0.1 + 0.3 * (1 - accuracy)  # 10% base + up to 30% based on poor accuracy
    neutral_prob = 1 - positive_prob - negative_prob
    
    feedback_type = np.random.choice(['positive', 'neutral', 'negative'], 
                                   p=[positive_prob, neutral_prob, negative_prob])
    
    feedback_score = {'positive': 1, 'neutral': 0, 'negative': -1}[feedback_type]
    
    # Simulate text feedback
    feedback_texts = {
        'positive': ["Very helpful!", "Exactly what I needed", "Great response", "Clear and accurate"],
        'neutral': ["OK response", "Adequate", "Could be better", "Standard answer"],
        'negative': ["Not helpful", "Doesn't answer my question", "Confusing", "Wrong information"]
    }
    
    feedback_text = np.random.choice(feedback_texts[feedback_type])
    
    feedback_data.append({
        'question_idx': idx,
        'category': row['category'],
        'complexity': row['complexity'],
        'policy_accuracy': row['policy_accuracy'],
        'feedback_type': feedback_type,
        'feedback_score': feedback_score,
        'feedback_text': feedback_text
    })

feedback_df = pd.DataFrame(feedback_data)

print("HUMAN FEEDBACK ANALYSIS")
print("=" * 40)

# Overall feedback distribution
feedback_distribution = feedback_df['feedback_type'].value_counts()
print(f"\nFeedback Distribution:")
for feedback_type, count in feedback_distribution.items():
    percentage = count / len(feedback_df) * 100
    print(f"- {feedback_type}: {count} ({percentage:.1f}%)")

# Correlation between automated scores and human feedback
correlation = feedback_df['policy_accuracy'].corr(feedback_df['feedback_score'])
print(f"\nCorrelation between Policy Accuracy and Human Feedback: {correlation:.3f}")

# Feedback by category
print(f"\nFeedback by Category:")
category_feedback = feedback_df.groupby('category')['feedback_score'].mean()
for category, avg_score in category_feedback.items():
    print(f"- {category}: {avg_score:.2f} average score")

# Identify discrepancies (high automated score, low human feedback)
results_with_feedback = results_df.merge(feedback_df, left_index=True, right_on='question_idx')

discrepancies = results_with_feedback[
    (results_with_feedback['policy_accuracy'] >= 0.8) & 
    (results_with_feedback['feedback_score'] <= 0)
]

print(f"\nDISCREPANCIES (High Automated Score, Low Human Feedback):")
if len(discrepancies) > 0:
    print(f"Found {len(discrepancies)} discrepancies to investigate:")
    for idx, row in discrepancies.iterrows():
        print(f"- {row['category']}: {row['question'][:50]}...")
        print(f"  Policy Accuracy: {row['policy_accuracy']:.2f}, Human Feedback: {row['feedback_text']}")
else:
    print("No significant discrepancies found!")

# Success stories (high automated score, high human feedback)
successes = results_with_feedback[
    (results_with_feedback['policy_accuracy'] >= 0.8) & 
    (results_with_feedback['feedback_score'] >= 1)
]

print(f"\nSUCCESS STORIES (High Automated Score, Positive Human Feedback):")
print(f"Found {len(successes)} success stories:")
for category, count in successes['category'].value_counts().items():
    print(f"- {category}: {count} positive examples")

print(f"\nSample positive feedback:")
positive_feedback = feedback_df[feedback_df['feedback_type'] == 'positive']['feedback_text'].unique()[:3]
for feedback in positive_feedback:
    print(f"- \"{feedback}\"")

print(f"\nSample negative feedback:")
negative_feedback = feedback_df[feedback_df['feedback_type'] == 'negative']['feedback_text'].unique()[:3]
for feedback in negative_feedback:
    print(f"- \"{feedback}\"")

# Summary insights
print(f"\nKEY INSIGHTS:")
print(f"1. Overall customer satisfaction: {feedback_df['feedback_score'].mean():.2f}/1.0")
print(f"2. Best performing category: {category_feedback.idxmax()} ({category_feedback.max():.2f} avg)")
print(f"3. Worst performing category: {category_feedback.idxmin()} ({category_feedback.min():.2f} avg)")
print(f"4. Automated-Human correlation: {correlation:.3f} ({'Strong' if abs(correlation) > 0.7 else 'Moderate' if abs(correlation) > 0.4 else 'Weak'})")

display(results_with_feedback[['question', 'category', 'policy_accuracy', 'feedback_type', 'feedback_text']].head())


In [0]:
# Apply Python-based scoring to all three datasets
import pandas as pd
from textblob import TextBlob

# Booking Records: Simulate prediction and scoring
booking_sample = booking_records_df.limit(100).toPandas()
booking_sample['prediction'] = booking_sample['primary_contact_last_name']  # Simulate output
booking_sample['accuracy'] = (booking_sample['prediction'] == booking_sample['primary_contact_last_name']).astype(int)
booking_sample['sentiment'] = booking_sample['prediction'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# FAQ: Simulate prediction and scoring
faq_sample = faq_df.limit(100).toPandas()
faq_sample['prediction'] = faq_sample['answer']  # Simulate output
faq_sample['accuracy'] = (faq_sample['prediction'] == faq_sample['answer']).astype(int)
faq_sample['relevance'] = faq_sample.apply(
    lambda row: len(set(str(row['prediction']).split()) & set(str(row['question']).split())) / max(len(set(str(row['question']).split())), 1),
    axis=1
)
faq_sample['sentiment'] = faq_sample['prediction'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

# QA: Already scored in previous cells (qa_sample)

# Display samples
print("Booking Records Sample with Scores:")
display(booking_sample[['booking_id', 'primary_contact_last_name', 'prediction', 'accuracy', 'sentiment']].head(5))

print("FAQ Sample with Scores:")
display(faq_sample[['question', 'answer', 'prediction', 'accuracy', 'relevance', 'sentiment']].head(5))

print("QA Sample with Scores:")
display(qa_sample[['question', 'answer', 'prediction', 'accuracy', 'relevance', 'sentiment']].head(5))

%md
### How to Evaluate Different GenAI Components in MLflow

- **Booking Records:** Evaluate structured outputs for accuracy, compliance, and user satisfaction. Use code-based scores for business logic (e.g., policy compliance, fee calculation).
- **FAQ:** Evaluate relevance, helpfulness, and sentiment. Use LLM-based scorers for nuanced judgment.
- **QA:** Use direct answer sheet evaluation for accuracy, relevance, and sentiment. Combine LLM-based and code-based scores for comprehensive assessment.

For advanced component evaluation, see: [MLflow GenAI Evaluation Scoring](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/scoring.html)

%md
## 4. Human Feedback {#human-feedback}

Human feedback is essential for evaluating GenAI applications as it captures the user experience and subjective qualities that automated metrics may miss. MLflow 3+ provides comprehensive tools for collecting, analyzing, and integrating human feedback into your evaluation workflow.

### Understanding the Feedback Data Model

The MLflow feedback data model supports structured collection and analysis of human judgments:

```python
# Feedback record structure
feedback_record = {
    "trace_id": "unique_trace_identifier",
    "feedback_type": "thumbs_up_down",  # or "rating", "categorical", "text"
    "feedback_value": 1,  # 1 for positive, -1 for negative, or rating scale
    "feedback_text": "Response was helpful and accurate",
    "user_id": "user_123",
    "timestamp": "2024-01-15T10:30:00Z",
    "metadata": {
        "source": "web_ui",
        "session_id": "session_456"
    }
}
```

### Types of Human Feedback

#### 1. End User Feedback
Direct feedback from application users during normal usage:
- **Thumbs up/down**: Simple binary feedback
- **Star ratings**: 1-5 star quality ratings  
- **Free text**: Open-ended comments and suggestions
- **Categorical**: Predefined categories (helpful, accurate, polite, etc.)

#### 2. Expert Feedback
Structured feedback from domain experts or reviewers:
- **Quality assessments**: Detailed evaluation against criteria
- **Ground truth validation**: Confirming correct answers
- **Policy compliance**: Checking adherence to guidelines
- **Comparative evaluation**: Ranking multiple responses

#### 3. A/B Testing Feedback
Comparative feedback between different model versions:
- **Preference ratings**: Which response is better?
- **Specific criteria comparison**: Better accuracy, helpfulness, etc.
- **Conversion metrics**: Task completion rates

### End User Feedback Collection

#### Using the Databricks Review App

The Databricks Review App provides a streamlined interface for collecting structured feedback:

```python
import mlflow.genai.labeling as labeling

# Create a labeling session
session = labeling.create_labeling_session(
    name="Unity Airways Customer Satisfaction Review",
    description="Collect feedback on customer service responses",
    assigned_users=["reviewer1@company.com", "reviewer2@company.com"]
)

# Add traces to review
traces_to_review = mlflow.search_traces(
    filter_string="attributes.status = 'OK'",
    max_results=50
)

session.add_traces(traces_to_review)
```

#### Embedding Feedback in Applications

Collect feedback directly in your application interface:

```python
import mlflow

# Log feedback alongside traces
@mlflow.trace
def handle_user_feedback(trace_id, feedback_type, feedback_value, feedback_text=None):
    """Collect and log user feedback for a specific interaction."""
    
    feedback_data = {
        "feedback_type": feedback_type,
        "feedback_value": feedback_value,
        "feedback_text": feedback_text,
        "timestamp": time.time()
    }
    
    # Log feedback as trace metadata
    mlflow.log_metadata(feedback_data)
    
    return {"status": "feedback_recorded"}

# Example usage in a web app
# When user clicks thumbs up:
handle_user_feedback(
    trace_id="trace_123", 
    feedback_type="thumbs_up_down", 
    feedback_value=1,
    feedback_text="Very helpful response!"
)
```

#### Programmatic Feedback Collection

Collect feedback through APIs or batch processing:

```python
def collect_batch_feedback(traces_df, feedback_df):
    """Merge traces with collected feedback data."""
    
    # Join traces with feedback
    traces_with_feedback = traces_df.merge(
        feedback_df, 
        left_on='trace_id', 
        right_on='trace_id',
        how='left'
    )
    
    # Calculate feedback metrics
    feedback_metrics = {
        'positive_feedback_rate': (traces_with_feedback['feedback_value'] == 1).mean(),
        'negative_feedback_rate': (traces_with_feedback['feedback_value'] == -1).mean(),
        'avg_rating': traces_with_feedback['feedback_value'].mean(),
        'total_feedback_count': traces_with_feedback['feedback_value'].notna().sum()
    }
    
    return traces_with_feedback, feedback_metrics
```

### Analyzing Feedback Data

#### Feedback Analytics

```python
def analyze_feedback_patterns(feedback_data):
    """Analyze patterns in user feedback."""
    
    # Feedback distribution
    feedback_distribution = feedback_data['feedback_value'].value_counts()
    
    # Feedback by time period
    feedback_data['date'] = pd.to_datetime(feedback_data['timestamp']).dt.date
    feedback_trends = feedback_data.groupby('date')['feedback_value'].mean()
    
    # Common themes in text feedback
    negative_feedback = feedback_data[feedback_data['feedback_value'] == -1]['feedback_text']
    positive_feedback = feedback_data[feedback_data['feedback_value'] == 1]['feedback_text']
    
    return {
        'distribution': feedback_distribution,
        'trends': feedback_trends,
        'negative_themes': negative_feedback.tolist(),
        'positive_themes': positive_feedback.tolist()
    }
```

#### Correlating Feedback with Model Performance

```python
def correlate_feedback_with_scores(traces_with_feedback):
    """Find correlations between automated scores and human feedback."""
    
    # Extract automated scores
    score_columns = [col for col in traces_with_feedback.columns if 'score' in col.lower()]
    
    correlations = {}
    for score_col in score_columns:
        correlation = traces_with_feedback[score_col].corr(traces_with_feedback['feedback_value'])
        correlations[score_col] = correlation
    
    # Identify which automated scores best predict human satisfaction
    best_predictors = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)
    
    return correlations, best_predictors
```

### Feedback-Driven Improvements

#### Using Feedback for Model Training

```python
def create_training_data_from_feedback(traces_with_feedback):
    """Convert feedback into training examples."""
    
    # Create preference pairs for RLHF
    positive_examples = traces_with_feedback[traces_with_feedback['feedback_value'] == 1]
    negative_examples = traces_with_feedback[traces_with_feedback['feedback_value'] == -1]
    
    training_data = []
    
    for _, pos_example in positive_examples.iterrows():
        for _, neg_example in negative_examples.iterrows():
            # Create preference pair
            training_data.append({
                'input': pos_example['input'],
                'chosen': pos_example['output'],
                'rejected': neg_example['output'],
                'preference_strength': abs(pos_example['feedback_value'] - neg_example['feedback_value'])
            })
    
    return training_data
```

#### Feedback-Based Dataset Curation

```python
def curate_dataset_from_feedback(traces_with_feedback, min_rating=4):
    """Create high-quality dataset from well-rated examples."""
    
    # Filter for high-quality examples
    high_quality = traces_with_feedback[traces_with_feedback['feedback_value'] >= min_rating]
    
    # Create evaluation dataset
    curated_examples = []
    for _, row in high_quality.iterrows():
        curated_examples.append({
            'inputs': row['input'],
            'expected': {
                'expected_response': row['output'],
                'quality_score': row['feedback_value']
            }
        })
    
    return curated_examples
```

Let's now demonstrate human feedback collection and analysis with our Unity Airways example.

In [0]:
import numpy as np

# Simulate feedback for 100 QA samples
np.random.seed(42)
feedback_options = ['positive', 'neutral', 'negative']
feedback_scores = {'positive': 1, 'neutral': 0, 'negative': -1}

qa_sample['user_feedback'] = np.random.choice(feedback_options, size=len(qa_sample))
qa_sample['feedback_score'] = qa_sample['user_feedback'].map(feedback_scores)

# Optionally, simulate free-text comments
comments = [None, "Great answer!", "Needs more detail.", "Too vague.", "Very helpful.", "Incorrect info."]
qa_sample['feedback_comment'] = np.random.choice(comments, size=len(qa_sample))

display(qa_sample[['question', 'answer', 'prediction', 'user_feedback', 'feedback_score', 'feedback_comment']].head(10))

In [0]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Create interactive feedback visualizations using Plotly
feedback_counts = qa_sample['user_feedback'].value_counts()

# Create subplots for feedback visualizations
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribution of User Feedback', 'Distribution of Feedback Scores'),
    specs=[[{"type": "bar"}, {"type": "histogram"}]]
)

# Feedback type distribution (bar chart)
colors_map = {'positive': 'green', 'neutral': 'gray', 'negative': 'red'}
colors = [colors_map.get(feedback_type, 'blue') for feedback_type in feedback_counts.index]

fig.add_trace(
    go.Bar(
        x=feedback_counts.index,
        y=feedback_counts.values,
        name='Feedback Count',
        marker_color=colors,
        showlegend=False,
        text=feedback_counts.values,
        textposition='auto',
        hovertemplate='<b>%{x}</b><br>Count: %{y}<extra></extra>'
    ),
    row=1, col=1
)

# Feedback score distribution (histogram)
fig.add_trace(
    go.Histogram(
        x=qa_sample['feedback_score'],
        nbinsx=3,
        name='Feedback Score',
        marker_color='skyblue',
        opacity=0.7,
        showlegend=False,
        hovertemplate='Score: %{x}<br>Count: %{y}<extra></extra>'
    ),
    row=1, col=2
)

# Update layout
fig.update_xaxes(title_text="Feedback Type", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_xaxes(title_text="Feedback Score", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    height=400,
    title_text="Human Feedback Analysis - Interactive Visualization",
    showlegend=False,
    hovermode='closest'
)

fig.show()

# Show sample comments
display(qa_sample[['question', 'prediction', 'user_feedback', 'feedback_comment']].head(10))

%md
### Best Practices for Human Feedback Collection

- **Integrate feedback collection into chatbot UI** to capture real user sentiment and suggestions.
- **Use structured feedback forms** (e.g., thumbs up/down, star ratings, free text) for actionable insights.
- **Analyze feedback regularly** to identify trends, common issues, and opportunities for improvement.
- **Correlate feedback with model outputs** to understand which responses drive positive or negative user experiences.
- **Leverage MLflow tracking** to log feedback data alongside evaluation runs for comprehensive analysis.
- **Use the Databricks Review app** for scalable feedback collection and annotation. See: [Databricks Review App Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/feedback.html)

Human feedback is a critical component in building trustworthy and effective GenAI applications.

%md
## 5. Evaluation Runs {#evaluation-runs}

Evaluation runs are the execution engine of GenAI evaluation in MLflow. They systematically apply scorers to evaluation datasets, track results, and enable comparison across model versions and time periods.

### Understanding Evaluation Runs

An evaluation run in MLflow 3+ consists of:

1. **Input Dataset**: The evaluation dataset containing test examples
2. **Model/Function**: The GenAI application being evaluated  
3. **Scorers**: The set of evaluation metrics to apply
4. **Execution Context**: Environment, parameters, and metadata
5. **Results**: Scores, traces, and artifacts generated

### Creating Evaluation Runs

#### Basic Evaluation Run

```python
import mlflow.genai

# Define your model function
@mlflow.trace
def customer_service_agent(question: str) -> dict:
    # Your GenAI application logic
    response = generate_response(question)
    return {"response": response}

# Define scorers
scorers = [
    RelevanceToQuery(),
    Safety(),
    Guidelines(name="professional_tone", guidelines="Use professional language")
]

# Run evaluation
eval_results = mlflow.genai.evaluate(
    data=evaluation_dataset,
    predict_fn=customer_service_agent,
    scorers=scorers,
    run_name="Unity Airways Customer Service v1.0"
)
```

#### Advanced Evaluation Configuration

```python
# More sophisticated evaluation with custom configuration
eval_results = mlflow.genai.evaluate(
    data=evaluation_dataset,
    predict_fn=customer_service_agent,
    scorers=scorers,
    run_name="Unity Airways v2.0 - Safety Enhanced",
    
    # Evaluation parameters
    evaluator_config={
        "timeout": 120,  # Timeout per example in seconds
        "max_workers": 5,  # Parallel evaluation workers
        "retry_attempts": 3  # Retry failed evaluations
    },
    
    # Additional metadata
    extra_metrics={
        "model_version": "2.0",
        "dataset_version": "1.2",
        "evaluation_date": "2024-01-15"
    }
)
```

### Evaluation Run Lifecycle

#### 1. Pre-Evaluation Setup

```python
# Set up MLflow tracking
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Unity-Airways-Evaluation")

# Configure evaluation environment
import os
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "10"

# Load and validate dataset
eval_dataset = mlflow.genai.datasets.load_dataset("catalog.schema.eval_dataset")
print(f"Loaded {len(eval_dataset)} examples for evaluation")
```

#### 2. During Evaluation

```python
# Monitor evaluation progress
with mlflow.start_run(run_name="Unity Airways Evaluation") as run:
    # Log pre-evaluation metadata
    mlflow.log_params({
        "dataset_size": len(eval_dataset),
        "model_version": "v2.0",
        "scorer_count": len(scorers)
    })
    
    # Run evaluation
    eval_results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=customer_service_agent,
        scorers=scorers
    )
    
    # Log additional artifacts
    mlflow.log_artifact("evaluation_config.yaml")
    mlflow.log_text(str(eval_results.summary), "evaluation_summary.txt")
```

#### 3. Post-Evaluation Analysis

```python
# Access evaluation results
print(f"Run ID: {eval_results.run_id}")
print(f"Evaluation completed with {len(eval_results.traces)} traces")

# Get detailed traces
evaluation_traces = mlflow.search_traces(run_id=eval_results.run_id)
print(f"Retrieved {len(evaluation_traces)} detailed traces")
```

### View and Interpret Results

#### MLflow UI Analysis

```python
# Generate URL to view results in MLflow UI
def get_evaluation_url(run_id):
    tracking_uri = mlflow.get_tracking_uri()
    return f"{tracking_uri}/#/experiments/{mlflow.get_experiment_by_name('Unity Airways').experiment_id}/runs/{run_id}"

print(f"View results: {get_evaluation_url(eval_results.run_id)}")
```

#### Programmatic Results Analysis

```python
def analyze_evaluation_results(eval_results):
    """Comprehensive analysis of evaluation results."""
    
    # Get run metrics
    run = mlflow.get_run(eval_results.run_id)
    metrics = run.data.metrics
    
    # Aggregate score analysis
    score_summary = {}
    for metric_name, value in metrics.items():
        if '/mean' in metric_name:
            scorer_name = metric_name.replace('metrics/', '').replace('/mean', '')
            score_summary[scorer_name] = value
    
    # Identify top and bottom performers
    sorted_scores = sorted(score_summary.items(), key=lambda x: x[1])
    worst_scorer = sorted_scores[0] if sorted_scores else None
    best_scorer = sorted_scores[-1] if sorted_scores else None
    
    # Calculate overall performance
    overall_score = sum(score_summary.values()) / len(score_summary) if score_summary else 0
    
    return {
        'overall_score': overall_score,
        'individual_scores': score_summary,
        'best_performer': best_scorer,
        'worst_performer': worst_scorer,
        'total_examples': len(eval_results.traces) if hasattr(eval_results, 'traces') else 0
    }

# Analyze results
analysis = analyze_evaluation_results(eval_results)
print(f"Overall Score: {analysis['overall_score']:.3f}")
print(f"Best Scorer: {analysis['best_performer']}")
print(f"Worst Scorer: {analysis['worst_performer']}")
```

#### Failure Analysis

```python
def analyze_failures(evaluation_traces):
    """Identify and analyze failed evaluations."""
    
    failures = []
    
    for trace in evaluation_traces:
        # Check for evaluation failures
        if hasattr(trace, 'assessments'):
            for assessment in trace.assessments:
                if assessment.feedback.value == 'no' or assessment.score < 0.5:
                    failures.append({
                        'trace_id': trace.trace_id,
                        'input': trace.request,
                        'output': trace.response,
                        'failed_scorer': assessment.name,
                        'score': assessment.score,
                        'rationale': assessment.rationale
                    })
    
    # Group failures by scorer
    failures_by_scorer = {}
    for failure in failures:
        scorer = failure['failed_scorer']
        if scorer not in failures_by_scorer:
            failures_by_scorer[scorer] = []
        failures_by_scorer[scorer].append(failure)
    
    return failures_by_scorer

# Analyze failures
failures = analyze_failures(evaluation_traces)
for scorer, failed_examples in failures.items():
    print(f"\n{scorer}: {len(failed_examples)} failures")
    if failed_examples:
        print(f"Example failure: {failed_examples[0]['rationale'][:100]}...")
```

### Comparing Evaluation Runs

#### Version Comparison

```python
def compare_evaluation_runs(run_id_1, run_id_2, run_name_1="V1", run_name_2="V2"):
    """Compare two evaluation runs."""
    
    # Get run data
    run_1 = mlflow.get_run(run_id_1)
    run_2 = mlflow.get_run(run_id_2)
    
    # Extract metrics
    metrics_1 = {k.replace('metrics/', ''): v for k, v in run_1.data.metrics.items() if '/mean' in k}
    metrics_2 = {k.replace('metrics/', ''): v for k, v in run_2.data.metrics.items() if '/mean' in k}
    
    # Calculate improvements
    comparison = []
    for metric in set(metrics_1.keys()) & set(metrics_2.keys()):
        improvement = metrics_2[metric] - metrics_1[metric]
        comparison.append({
            'metric': metric.replace('/mean', ''),
            f'{run_name_1}_score': metrics_1[metric],
            f'{run_name_2}_score': metrics_2[metric],
            'improvement': improvement,
            'improvement_pct': (improvement / metrics_1[metric] * 100) if metrics_1[metric] != 0 else 0
        })
    
    return pd.DataFrame(comparison)

# Compare runs
comparison_df = compare_evaluation_runs(eval_results_v1.run_id, eval_results_v2.run_id)
display(comparison_df)
```

#### Trend Analysis

```python
def analyze_evaluation_trends(experiment_name, days=30):
    """Analyze evaluation trends over time."""
    
    # Get experiment
    experiment = mlflow.get_experiment_by_name(experiment_name)
    
    # Search runs from last N days
    cutoff_time = int((time.time() - days * 24 * 60 * 60) * 1000)
    
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"attributes.start_time > {cutoff_time}",
        order_by=["attributes.start_time DESC"]
    )
    
    # Extract trend data
    trend_data = []
    for _, run in runs.iterrows():
        metrics = {k: v for k, v in run.items() if k.startswith('metrics.') and '/mean' in k}
        trend_data.append({
            'run_id': run['run_id'],
            'run_name': run['tags.mlflow.runName'],
            'start_time': pd.to_datetime(run['start_time']),
            **metrics
        })
    
    return pd.DataFrame(trend_data)

# Analyze trends
trends_df = analyze_evaluation_trends("Unity Airways Evaluation")
display(trends_df.head())
```

Let's now demonstrate these evaluation run concepts with our Unity Airways customer service agent.

In [0]:
# Aggregate key metrics from the QA sample
summary = qa_sample[['accuracy', 'relevance', 'sentiment', 'length_score', 'feedback_score']].mean().to_frame('mean').T
summary['total_samples'] = len(qa_sample)
display(summary)

# Show distribution of feedback types
feedback_dist = qa_sample['user_feedback'].value_counts().to_frame('count')
display(feedback_dist)

%md
### How to Log and View Evaluation Runs in MLflow 3+

If your workspace supports MLflow 3+ GenAI evaluation APIs, you can log, view, and compare evaluation runs using the MLflow tracking UI and APIs.

**Example (Python):**
```python
import mlflow
with mlflow.start_run(run_name="Unity Airways QA Evaluation"):
    mlflow.log_metrics({
        "accuracy": summary['accuracy'].values[0],
        "relevance": summary['relevance'].values[0],
        "sentiment": summary['sentiment'].values[0],
        "length_score": summary['length_score'].values[0],
        "feedback_score": summary['feedback_score'].values[0],
        "total_samples": summary['total_samples'].values[0]
    })
    # Optionally log artifacts, charts, or feedback data
```

For more details, see: [MLflow GenAI Evaluation Runs](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)

If MLflow 3+ APIs are not available, use Python to aggregate and review results as shown above.

%md
### MLflow 3+ API Usage and QA Dataset Focus

- All MLflow 3+ API examples are retained in this notebook for reference and reproducibility.
- Evaluation and scoring steps are focused on the QA dataset, which contains ground truth labels for robust GenAI assessment.
- MLflow 3+ APIs are used for scoring, logging, and tracking evaluation runs for the QA dataset.
- For more details, see the official documentation: [MLflow GenAI Evaluation Scoring](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/scoring.html)

%md
## Step 3: Keep All MLflow 3+ API Examples and Focus Evaluation on QA Dataset
I will ensure that all MLflow 3+ API examples are present and clearly marked, and that the evaluation and scoring steps focus on the QA dataset. I will add a markdown cell summarizing this, and a code cell showing MLflow 3+ scoring and logging for the QA dataset.

%md
## 6. Best Practices {#best-practices}

This section consolidates the key best practices for evaluating GenAI applications with MLflow 3+ on Databricks, drawn from both this demonstration and production experience.

### Dataset Management Best Practices

#### 1. Dataset Diversity and Quality
```python
# Ensure diverse coverage across use cases
dataset_coverage_check = {
    'booking_scenarios': ['new_booking', 'modification', 'cancellation'],
    'policy_questions': ['baggage', 'refund', 'change_fees'],
    'customer_types': ['first_time', 'frequent_flyer', 'business'],
    'complexity_levels': ['simple', 'multi_part', 'edge_case']
}

# Validate dataset balance
def validate_dataset_balance(dataset):
    """Ensure balanced representation across key dimensions."""
    # Implementation for checking dataset balance
    pass
```

#### 2. Version Control and Lineage
- Use MLflow's managed datasets for version tracking
- Document dataset changes and rationale
- Maintain lineage between datasets and evaluation runs
- Regular dataset refresh from production data

#### 3. Ground Truth Management
```python
# Establish ground truth validation process
def validate_ground_truth(examples):
    """Multi-reviewer validation for ground truth examples."""
    for example in examples:
        # Require multiple expert reviews for contentious examples
        if example['confidence_score'] < 0.8:
            example['requires_review'] = True
    return examples
```

### Scorer Selection and Configuration

#### 1. Balanced Scorer Portfolio
```python
# Recommended scorer mix for customer service
comprehensive_scorers = [
    # Factual correctness
    RetrievalGroundedness(),
    AnswerCorrectness(),
    
    # User experience  
    RelevanceToQuery(),
    Guidelines(name="helpfulness", guidelines="Response is helpful and actionable"),
    
    # Safety and compliance
    Safety(),
    Guidelines(name="brand_compliance", guidelines="Follows company brand guidelines"),
    
    # Technical quality
    CodeBasedScorer(name="response_length", func=length_checker),
    CodeBasedScorer(name="contact_info", func=contact_info_checker)
]
```

#### 2. Scorer Calibration
- Regularly validate LLM judges against human judgment
- Monitor scorer consistency across evaluation runs
- Adjust guidelines based on performance analysis

#### 3. Custom Scorer Development
```python
# Template for robust custom scorers
def create_robust_scorer(name, validation_func, fallback_score=0.5):
    """Create a robust scorer with error handling."""
    
    def scorer_func(prediction, **kwargs):
        try:
            return validation_func(prediction, **kwargs)
        except Exception as e:
            # Log error and return fallback
            mlflow.log_metric(f"{name}_errors", 1)
            return {"score": fallback_score, "justification": f"Scorer error: {str(e)}"}
    
    return CodeBasedScorer(name=name, func=scorer_func)
```

### Human Feedback Integration

#### 1. Multi-Channel Feedback Collection
```python
# Comprehensive feedback strategy
feedback_channels = {
    'inline_ui': 'Thumbs up/down in application',
    'periodic_surveys': 'Detailed user satisfaction surveys',
    'expert_review': 'Domain expert evaluation sessions',
    'a_b_testing': 'Comparative evaluation of versions'
}
```

#### 2. Feedback Quality Assurance
- Validate feedback consistency across reviewers
- Filter out low-quality or biased feedback
- Implement inter-rater reliability checks

#### 3. Feedback-Driven Improvement Loop
```python
def implement_feedback_loop():
    """Systematic feedback integration process."""
    
    # 1. Collect feedback
    feedback_data = collect_user_feedback()
    
    # 2. Analyze patterns
    issues = identify_common_issues(feedback_data)
    
    # 3. Update evaluation criteria
    new_scorers = create_scorers_for_issues(issues)
    
    # 4. Retrain/adjust model
    model_updates = generate_model_improvements(issues)
    
    # 5. Validate improvements
    validation_results = run_evaluation_with_feedback(new_scorers)
    
    return validation_results
```

### Evaluation Run Management

#### 1. Systematic Evaluation Cadence
```python
# Establish regular evaluation schedule
evaluation_schedule = {
    'daily': 'Quick smoke tests on key scenarios',
    'weekly': 'Comprehensive evaluation on full dataset',
    'monthly': 'Deep analysis with human feedback integration',
    'release': 'Full evaluation before production deployment'
}
```

#### 2. Regression Testing
```python
def regression_test_framework(baseline_run_id, new_run_id):
    """Systematic regression testing between versions."""
    
    comparison = compare_evaluation_runs(baseline_run_id, new_run_id)
    
    # Define regression thresholds
    regression_thresholds = {
        'safety': -0.01,  # No degradation in safety
        'relevance': -0.05,  # Max 5% degradation in relevance
        'groundedness': -0.02  # Max 2% degradation in factual accuracy
    }
    
    regressions = []
    for _, row in comparison.iterrows():
        metric = row['metric']
        improvement = row['improvement']
        
        if metric in regression_thresholds:
            if improvement < regression_thresholds[metric]:
                regressions.append({
                    'metric': metric,
                    'degradation': improvement,
                    'threshold': regression_thresholds[metric]
                })
    
    return regressions
```

#### 3. Performance Monitoring
```python
# Monitor evaluation run performance
def monitor_evaluation_performance():
    """Track evaluation system health."""
    
    metrics_to_track = [
        'evaluation_runtime',
        'scorer_success_rate',
        'trace_generation_rate',
        'feedback_collection_rate'
    ]
    
    # Alert on performance degradation
    for metric in metrics_to_track:
        current_value = get_current_metric(metric)
        baseline_value = get_baseline_metric(metric)
        
        if current_value < baseline_value * 0.8:  # 20% degradation threshold
            send_alert(f"{metric} degraded: {current_value} vs {baseline_value}")
```

### Production Deployment Best Practices

#### 1. Staged Evaluation Deployment
```python
# Multi-stage evaluation deployment
deployment_stages = {
    'development': 'Full evaluation suite for development',
    'staging': 'Production-like evaluation with synthetic data',
    'canary': 'Limited production evaluation on small traffic',
    'production': 'Full production monitoring with sampling'
}
```

#### 2. Real-Time Monitoring
```python
@mlflow.trace
def production_service_with_monitoring(user_query):
    """Production service with integrated evaluation."""
    
    # Generate response
    response = generate_customer_service_response(user_query)
    
    # Real-time safety check
    safety_score = quick_safety_check(response)
    if safety_score < 0.8:
        # Fallback to safe response
        response = get_safe_fallback_response(user_query)
        mlflow.log_metric("safety_fallback_triggered", 1)
    
    # Log for offline evaluation
    mlflow.log_metadata({
        "user_query": user_query,
        "response": response,
        "safety_score": safety_score
    })
    
    return response
```

#### 3. Continuous Learning
```python
def continuous_improvement_pipeline():
    """Automated pipeline for continuous model improvement."""
    
    # 1. Collect recent production data
    recent_traces = collect_recent_production_traces(days=7)
    
    # 2. Run evaluation on production data
    production_eval = run_evaluation_on_traces(recent_traces)
    
    # 3. Compare with baseline
    performance_delta = compare_with_baseline(production_eval)
    
    # 4. If degradation detected, trigger improvement workflow
    if performance_delta < -0.05:  # 5% degradation threshold
        trigger_model_improvement_workflow(recent_traces, production_eval)
    
    # 5. Update evaluation datasets with high-quality examples
    update_datasets_with_production_examples(recent_traces)
```

### Organizational Best Practices

#### 1. Cross-Functional Collaboration
- Involve domain experts in scorer development
- Regular review sessions with product and engineering teams
- Shared ownership of evaluation quality

#### 2. Documentation and Knowledge Sharing
```python
# Maintain comprehensive evaluation documentation
evaluation_documentation = {
    'scorer_definitions': 'Clear definition of each evaluation metric',
    'dataset_descriptions': 'Purpose and composition of each dataset',
    'baseline_performance': 'Historical performance benchmarks',
    'improvement_playbooks': 'Standard procedures for addressing issues'
}
```

#### 3. Training and Onboarding
- Regular training on evaluation best practices
- Hands-on workshops with MLflow 3+ evaluation tools
- Knowledge sharing sessions on evaluation insights

### Summary

This notebook demonstrates a comprehensive approach to evaluating GenAI applications using MLflow 3+ on Databricks. Key takeaways include:

1. **Holistic Evaluation Strategy**: Combine automated scoring with human feedback
2. **Systematic Dataset Management**: Use diverse, version-controlled evaluation datasets
3. **Balanced Scorer Portfolio**: Mix LLM-based and code-based scorers for comprehensive assessment
4. **Continuous Improvement**: Implement feedback loops for ongoing enhancement
5. **Production Integration**: Monitor and evaluate in production environments
6. **Cross-Functional Collaboration**: Involve domain experts and stakeholders

By following these practices, organizations can build robust, reliable GenAI applications that consistently deliver value to users while maintaining safety and quality standards.

### Next Steps

1. **Implement Evaluation Pipeline**: Set up automated evaluation runs for your GenAI application
2. **Establish Feedback Collection**: Integrate human feedback collection into your application
3. **Create Custom Scorers**: Develop domain-specific evaluation metrics
4. **Monitor Production Performance**: Implement real-time quality monitoring
5. **Build Improvement Processes**: Create systematic workflows for addressing evaluation insights

For additional resources and detailed documentation, visit:
- [MLflow GenAI Evaluation Documentation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)
- [Databricks GenAI Best Practices](https://docs.databricks.com/aws/en/mlflow3/genai/)
- [MLflow 3+ Release Notes](https://docs.databricks.com/release-notes/mlflow/index.html)